#Medallion Architecture

Medallion Architecture is a data design pattern used in Databricks Lakehouse to organize data into three layers: **Bronze, Silver and Gold.**



![](/Workspace/Users/rinkimehra068@gmail.com/Multihop Demo/Multihop.png)

###1. Bronze Layer – Raw Data
- Stores data directly from external source systems.
- Data is stored in its original structure.
- Basic metadata is added, such as ingestion timestamp and source file name, path.

In [0]:
dbutils.widgets.text("catalog", "dev")
catalog = dbutils.widgets.get("catalog")

In [0]:
#ingestion to bronze using CTAS with metadata columns
spark.sql(f"""
create table if not exists {catalog}.bronze.sales as
select *,
        _metadata.file_name as _source_name,
        _metadata.file_path as _source_path,
        current_timestamp() as _ingested_at
from read_files("/Volumes/{catalog}/bronze/raw/sales/sales.csv")
""")


In [0]:
%sql
select * from dev.bronze.sales limit 5

###2. Silver Layer – Cleaned and Refined Data
- Cleans and validates Bronze data.
- Removes duplicate and invalid records.
- Handles missing and incorrect values.


In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window
import pyspark.sql.functions as F

bronze_sales = spark.read.table(f"{catalog}.bronze.sales")

#Handling Duplicates
w = Window.partitionBy("order_id").orderBy(F.col("order_date").desc())
df_ranked = bronze_sales.withColumn("_rn", F.row_number().over(w))

#Handling triming, casing and type casting
tagged = (
    df_ranked
    .withColumn("order_id", trim(upper(col("order_id"))))
    .withColumn("customer_id", trim(upper(col("customer_id"))))
    .withColumn("transaction_id", trim(upper(col("transaction_id"))))
    .withColumn("product_id", trim(upper(col("product_id"))))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("discount_amount", col("discount_amount").cast("double"))
    .withColumn("total_amount", col("total_amount").cast("double"))
    .withColumn("order_date", to_date(col("order_date")))
)

#Handle nulls and negative quantity, discount amount and total_amount
# (rows with nulls in these fields are routed to quarantine instead of silently dropped)
is_invalid = (
    F.col("quantity").isNull()
    | F.col("discount_amount").isNull()
    | F.col("total_amount").isNull()
    | (F.col("quantity") < 0)
    | (F.col("discount_amount") < 0)
    | (F.col("total_amount") < 0)
)

#clean_df keeps only the latest record per order_id (_rn == 1) and valid values
#quarantine_df holds BOTH invalid records AND the duplicate (older) records per order_id (_rn > 1)
clean_df = tagged.filter((F.col("_rn") == 1) & ~is_invalid).drop("_rn")
quarantine_df = tagged.filter(is_invalid | (F.col("_rn") > 1)).drop("_rn")


# 5. Write silver tables (full refresh of the batch)
clean_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.sales_clean")

quarantine_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.silver.sales_quarantine")

In [0]:
%sql
select count(*) from dev.silver.sales_clean

In [0]:
%sql
select count(*) from dev.silver.sales_quarantine

###3. Gold Layer – Business-Ready Data
- Contains business-level data.
- Creates aggregated and optimized datasets.
- Supports dashboards, reporting, analytics and machine learning.

In [0]:
# 1. Total sales across all orders
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.total_sales AS
SELECT sum(total_amount) AS total_sales
FROM {catalog}.silver.sales_clean
""")

display(spark.table(f"{catalog}.gold.total_sales"))

In [0]:
# 2. Total number of orders
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.total_orders AS
SELECT count(DISTINCT order_id) AS total_orders
FROM {catalog}.silver.sales_clean
""")

display(spark.table(f"{catalog}.gold.total_orders"))

In [0]:
#3. Total sales for each customer
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.gold.sales_by_customer AS
SELECT customer_id, sum(total_amount) AS total_sales
FROM {catalog}.silver.sales_clean
GROUP BY customer_id
""")

display(spark.table(f"{catalog}.gold.sales_by_customer"))